# Yelp RAG Agent — LMDeploy Deployment Benchmark

**Hardware:** Colab A100 80GB  
**Goal:** Compare LMDeploy fp16 vs AWQ on the same hardware under identical conditions.

| Experiment | Model | Backend |
|---|---|---|
| `lmdeploy_fp16` | Qwen2.5-7B-Instruct (fp16) | pytorch |
| `lmdeploy_awq`  | Qwen2.5-7B-Instruct-AWQ (int4) | turbomind |

Results are saved to `results/deployment_perf.json` and Google Drive.

---
**Before running:** Set `GDRIVE_RESULTS_DIR` in Cell 2 to your Drive path.

## Cell 1 — Environment setup

In [ ]:
# Verify A100 is available
!nvidia-smi

# Install LMDeploy and project dependencies
!pip install lmdeploy -q
!pip install pyyaml requests -q

print("\nEnvironment ready.")

## Cell 2 — Mount Google Drive and clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Configure these paths ────────────────────────────────────────
GDRIVE_RESULTS_DIR  = "/content/drive/MyDrive/yelp_rag_results"  # where to save outputs
GDRIVE_VECTORSTORE  = "/content/drive/MyDrive/yelp_rag_vectorstore"  # vectorstore files
GDRIVE_DATA         = "/content/drive/MyDrive/yelp_rag_data"         # CSV + business JSON
# ─────────────────────────────────────────────────────────────────

import os
os.makedirs(GDRIVE_RESULTS_DIR, exist_ok=True)

print(f"Drive mounted. Results will be saved to: {GDRIVE_RESULTS_DIR}")

## Cell 3 — Clone project and link data files

In [ ]:
import os

REPO_URL  = "https://github.com/your-username/yelp-rag-agent-deployment"  # ← update
REPO_DIR  = "/content/yelp-rag-agent-deployment"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

# Install package (no heavy deps needed for metrics module)
!pip install -e . --no-deps -q

# Link large files from Drive to avoid re-downloading
import subprocess

def symlink_if_exists(src, dst):
    if os.path.exists(src) and not os.path.exists(dst):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        os.symlink(src, dst)
        print(f"  Linked: {src} → {dst}")
    elif not os.path.exists(src):
        print(f"  WARNING: source not found: {src}")

symlink_if_exists(f"{GDRIVE_VECTORSTORE}/review_chunks.index",
                  f"{REPO_DIR}/vectorstore/review_chunks.index")
symlink_if_exists(f"{GDRIVE_VECTORSTORE}/review_chunks.pkl",
                  f"{REPO_DIR}/vectorstore/review_chunks.pkl")
symlink_if_exists(f"{GDRIVE_DATA}/yelp_reviews_sampled_50k.csv",
                  f"{REPO_DIR}/data/processed/yelp_reviews_sampled_50k.csv")
symlink_if_exists(f"{GDRIVE_DATA}/yelp_academic_dataset_business.json",
                  f"{REPO_DIR}/data/raw/yelp_academic_dataset_business.json")

print("\nSetup complete.")

## Cell 4 — Helper functions

In [ ]:
import subprocess, time, sys, json, os, requests
from pathlib import Path
from typing import Optional

# ── VRAM helpers ─────────────────────────────────────────────────

def get_vram_gb() -> dict:
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.used,memory.total",
             "--format=csv,noheader,nounits"],
            stderr=subprocess.DEVNULL,
        ).decode().strip()
        used, total = map(int, out.split(", "))
        return {"used_gb": round(used/1024, 2), "total_gb": round(total/1024, 2)}
    except Exception as e:
        return {"used_gb": None, "total_gb": None, "error": str(e)}

# ── Server helpers ────────────────────────────────────────────────

BASE_URL    = "http://localhost:23333"
PORT        = 23333
all_results = []

# LMDeploy server configs — no shell script needed
SERVER_CONFIGS = {
    "fp16": {
        "model":   "Qwen/Qwen2.5-7B-Instruct",
        "backend": "pytorch",
        "extra":   [],
    },
    "awq": {
        "model":   "Qwen/Qwen2.5-7B-Instruct-AWQ",
        "backend": "turbomind",
        "extra":   ["--model-format", "awq"],
    },
}

def wait_for_server(base_url: str, timeout: int = 600, poll_interval: int = 10) -> bool:
    """Poll /v1/models until server is ready (default 10 min for first-run downloads)."""
    deadline = time.time() + timeout
    dots = 0
    while time.time() < deadline:
        try:
            if requests.get(f"{base_url}/v1/models", timeout=5).status_code == 200:
                print()
                return True
        except Exception:
            pass
        time.sleep(poll_interval)
        dots += 1
        print(".", end="", flush=True)
        if dots % 6 == 0:
            elapsed = int(time.time() - (deadline - timeout))
            print(f" ({elapsed}s elapsed)", flush=True)
    print()
    return False

def get_server_model_name(base_url: str) -> str:
    """Query /v1/models and return the actual registered model name."""
    resp = requests.get(f"{base_url}/v1/models", timeout=10)
    models = resp.json().get("data", [])
    if not models:
        raise RuntimeError("No models registered on the server.")
    name = models[0]["id"]
    print(f"  [server] Registered model name: {name!r}")
    return name

def start_server(mode: str):
    cfg = SERVER_CONFIGS[mode]
    cmd = [
        "lmdeploy", "serve", "api_server",
        cfg["model"],
        "--server-port", str(PORT),
        "--backend", cfg["backend"],
    ] + cfg["extra"]

    log_path = f"/tmp/lmdeploy_{mode}.log"
    print(f"Starting LMDeploy server (mode={mode})")
    print(f"  Command: {' '.join(cmd)}")

    proc = subprocess.Popen(
        cmd,
        stdout=open(log_path, "w"),
        stderr=subprocess.STDOUT,
    )
    print(f"  PID: {proc.pid}")
    print("Waiting up to 10 minutes (model may need to download on first run) ...")

    if wait_for_server(BASE_URL, timeout=600):
        print("Server is ready.")
    else:
        print("\n[ERROR] Server did not become ready within 10 minutes.")
        print("Last server log:")
        print("-" * 60)
        try:
            with open(log_path) as f:
                content = f.read()
                print(content[-3000:] if len(content) > 3000 else content)
        except Exception:
            pass
        print("-" * 60)
        proc.terminate()
        raise RuntimeError(
            f"LMDeploy server (mode={mode}) failed to start. "
            "Check the log above for details."
        )
    return proc

def stop_server(proc):
    proc.terminate()
    proc.wait()
    time.sleep(15)
    print(f"Server stopped. VRAM now: {get_vram_gb()}")

# ── Metrics ───────────────────────────────────────────────────────

BENCHMARK_PROMPTS = [
    "What do customers commonly complain about at restaurants?",
    "Summarize the typical experience at a 1-star rated business.",
    "What aspects of service do customers mention most in negative reviews?",
    "Describe patterns in reviews that mention wait time.",
    "What do customers say about food quality in positive reviews?",
]

def measure_ttft(base_url: str, model: str,
                 prompt: str = "Hello, who are you?") -> float:
    t0 = time.perf_counter()
    try:
        with requests.post(
            f"{base_url}/v1/chat/completions",
            json={"model": model,
                  "messages": [{"role": "user", "content": prompt}],
                  "stream": True, "max_tokens": 10},
            stream=True, timeout=60,
        ) as resp:
            if resp.status_code != 200:
                print(f"  [TTFT] HTTP {resp.status_code}: {resp.text[:300]}")
                return -1.0
            for chunk in resp.iter_lines():
                if chunk and chunk not in (b"data: [DONE]", b""):
                    return round((time.perf_counter() - t0) * 1000, 1)
    except Exception as e:
        print(f"  [TTFT] Exception: {e}")
    return -1.0

def measure_throughput(base_url: str, model: str,
                       prompts: list, max_tokens: int = 200) -> dict:
    times, word_counts, errors = [], [], 0
    for i, prompt in enumerate(prompts):
        t0 = time.perf_counter()
        try:
            resp = requests.post(
                f"{base_url}/v1/chat/completions",
                json={"model": model,
                      "messages": [{"role": "user", "content": prompt}],
                      "max_tokens": max_tokens},
                timeout=120,
            )
            resp.raise_for_status()
            content = resp.json()["choices"][0]["message"]["content"]
            elapsed = time.perf_counter() - t0
            times.append(elapsed)
            word_counts.append(len(content.split()))
            print(f"  [throughput] prompt {i+1}/{len(prompts)}: {elapsed:.1f}s, "
                  f"{len(content.split())} words")
        except Exception as e:
            print(f"  [throughput] prompt {i+1} failed: {e}")
            errors += 1
    if not times:
        return {"avg_response_s": None, "estimated_tps": None,
                "n_samples": len(prompts), "errors": errors}
    return {
        "avg_response_s": round(sum(times)/len(times), 2),
        "estimated_tps":  round(sum(word_counts)/sum(times), 1),
        "n_samples":      len(prompts),
        "errors":         errors,
    }

def run_deployment_experiment(experiment_id, backend_label,
                               base_url="http://localhost:23333",
                               vram_baseline_gb=None) -> dict:
    print(f"\n[metrics] Running: {experiment_id}")

    model = get_server_model_name(base_url)

    vram_after = get_vram_gb()
    vram_load  = (round(vram_after["used_gb"] - vram_baseline_gb, 2)
                  if vram_baseline_gb is not None and vram_after["used_gb"] is not None
                  else vram_after["used_gb"])
    print(f"  [VRAM] after load: {vram_after['used_gb']} GB  (load delta: {vram_load} GB)")

    print("  [metrics] Measuring TTFT ...")
    ttft = measure_ttft(base_url, model)
    print(f"  [TTFT] {ttft} ms")

    print("  [metrics] Measuring throughput ...")
    tp = measure_throughput(base_url, model, BENCHMARK_PROMPTS)

    vram_peak = get_vram_gb()
    print(f"  [VRAM] peak after inference: {vram_peak['used_gb']} GB")

    result = {
        "experiment_id":  experiment_id,
        "model":          model,
        "backend":        backend_label,
        "vram_load_gb":   vram_load,
        "vram_peak_gb":   vram_peak.get("used_gb"),
        "ttft_ms":        ttft,
        "avg_response_s": tp["avg_response_s"],
        "estimated_tps":  tp["estimated_tps"],
        "n_samples":      tp["n_samples"],
        "errors":         tp["errors"],
    }
    print(f"\n[metrics] Result: {json.dumps(result, indent=2)}")
    return result

# ── Save helper ───────────────────────────────────────────────────

def save_results(results, path="results/deployment_perf.json",
                 hardware="Colab A100 80GB"):
    output = {"hardware": hardware, "experiments": results}
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2)
    print(f"[metrics] Saved to {path}")

print("Cell 4 loaded — all helpers ready.")

## Cell 5 — Baseline VRAM (before loading any model)

In [ ]:
vram_baseline = get_vram_gb()
print(f"Baseline VRAM: {vram_baseline}")

## Cell 6 — Experiment 1: LMDeploy + fp16

In [ ]:
# ── Start fp16 server ────────────────────────────────────────────
proc_fp16 = start_server("fp16")

# ── Run experiment ───────────────────────────────────────────────
result_fp16 = run_deployment_experiment(
    experiment_id    = "lmdeploy_fp16",
    backend_label    = "lmdeploy-pytorch",
    base_url         = BASE_URL,
    vram_baseline_gb = vram_baseline["used_gb"],
)
all_results.append(result_fp16)

# ── Stop server ──────────────────────────────────────────────────
stop_server(proc_fp16)

print("\nExperiment 1 complete.")
print(json.dumps(result_fp16, indent=2))

## Cell 7 — Experiment 2: LMDeploy + AWQ

In [ ]:
# ── Start AWQ server ─────────────────────────────────────────────
proc_awq = start_server("awq")

# ── Run experiment ───────────────────────────────────────────────
result_awq = run_deployment_experiment(
    experiment_id    = "lmdeploy_awq",
    backend_label    = "lmdeploy-turbomind",
    base_url         = BASE_URL,
    vram_baseline_gb = vram_baseline["used_gb"],
)
all_results.append(result_awq)

# ── Stop server ──────────────────────────────────────────────────
stop_server(proc_awq)

print("\nExperiment 2 complete.")
print(json.dumps(result_awq, indent=2))

## Cell 8 — Save results

In [ ]:
import shutil

LOCAL_PATH = f"{REPO_DIR}/results/deployment_perf.json"
DRIVE_PATH = f"{GDRIVE_RESULTS_DIR}/deployment_perf.json"

# Save locally
save_results(all_results, path=LOCAL_PATH, hardware="Colab A100 80GB")

# Backup to Drive
shutil.copy(LOCAL_PATH, DRIVE_PATH)
print(f"Backed up to Drive: {DRIVE_PATH}")

# Print summary table
print("\n" + "="*60)
print("DEPLOYMENT PERFORMANCE SUMMARY")
print("="*60)
header = f"{'Experiment':<20} {'VRAM Load':>10} {'VRAM Peak':>10} {'TTFT(ms)':>10} {'TPS':>8} {'Avg(s)':>8}"
print(header)
print("-" * len(header))
for r in all_results:
    print(f"{r['experiment_id']:<20} "
          f"{str(r['vram_load_gb'])+'GB':>10} "
          f"{str(r['vram_peak_gb'])+'GB':>10} "
          f"{str(r['ttft_ms'])+'ms':>10} "
          f"{str(r['estimated_tps']):>8} "
          f"{str(r['avg_response_s'])+'s':>8}")

## Cell 9 — Quality evaluation (separate notebook)

Quality evaluation (20-question set against fp16 and AWQ backends) has been\n",
    "moved to a dedicated notebook that requires the full package:\n",
    "\n",
    "**`notebooks/colab_quality_eval.ipynb`**\n",
    "\n",
    "Prerequisites before running that notebook:\n",
    "1. Push the repo to GitHub with the `src/yelp_rag_agent/` structure\n",
    "2. Upload vectorstore and data files to Google Drive\n",
    "3. The deployment benchmark in this notebook (Cells 5–8) has already run successfully